# Data profile: sales, weather and Kiel Week

**Purpose:** build a factual picture of the three raw files before any modelling: what one row means, which keys are unique, which period is covered, where values are missing and whether the joins keep every sales row. The results are summarised in `docs/data-fact-sheet.md` and block 5 of `CANVAS.md`; the decisions taken on them are in `DECISIONS.md`.

The raw data is only read, never changed.

In [1]:
import pandas as pd

BASE_URL = "https://raw.githubusercontent.com/opencampus-sh/intro-to-data-science-and-ml/main/data/bakery/"
sales = pd.read_csv(BASE_URL + "umsatzdaten_gekuerzt.csv", parse_dates=["Datum"])
weather = pd.read_csv(BASE_URL + "wetter.csv", parse_dates=["Datum"])
kiel_week = pd.read_csv(BASE_URL + "kiwo.csv", parse_dates=["Datum"])

GROUPS = {1: "Bread", 2: "Rolls", 3: "Croissants", 4: "Confectionery", 5: "Cakes", 6: "Seasonal bread"}
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

## 1. Sales: shape, key and period
**Grain** means what one row stands for. Here we expect one row per product group and day. A **key** is a column (or combination) that identifies a row uniquely; duplicated keys would mean double-counted turnover.

In [2]:
print("Shape:", sales.shape)
print("Missing values per column:")
print(sales.isna().sum())
print("\nid unique:", sales["id"].is_unique)
print("Duplicated (Datum, Warengruppe):", sales.duplicated(["Datum", "Warengruppe"]).sum())
id_expected = sales["Datum"].dt.strftime("%y%m%d").astype(int) * 10 + sales["Warengruppe"]
print("id = YYMMDD + group for every row:", (sales["id"] == id_expected).all())
print("Period:", sales["Datum"].min().date(), "to", sales["Datum"].max().date())

Shape: (9334, 4)
Missing values per column:
id             0
Datum          0
Warengruppe    0
Umsatz         0
dtype: int64

id unique: True
Duplicated (Datum, Warengruppe): 0
id = YYMMDD + group for every row: True
Period: 2013-07-01 to 2018-07-31


## 2. Turnover per product group
The groups differ strongly in level, so we describe each one separately. Outliers are counted with a wide rule (more than 3 inter-quartile ranges outside the middle half of the values). Decision: extreme values are kept as real demand.

In [3]:
stats = sales.groupby("Warengruppe")["Umsatz"].describe().round(1)


def count_outliers(x):
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((x > q3 + 3 * iqr) | (x < q1 - 3 * iqr)).sum()


stats["outliers_3iqr"] = sales.groupby("Warengruppe")["Umsatz"].apply(count_outliers)
stats.index = stats.index.map(GROUPS)
stats

,count,mean,std,min,25%,50%,75%,max,outliers_3iqr
Warengruppe,,,,,,,,,
Bread,1819.0,122.1,40.3,23.1,96.4,120.1,144.3,416.8,8
Rolls,1819.0,402.5,139.7,175.0,289.1,370.8,492.8,1203.4,1
Croissants,1819.0,163.3,75.6,37.7,105.6,142.8,202.0,565.9,1
Confectionery,1766.0,88.0,35.3,27.4,66.0,80.0,98.3,430.5,27
Cakes,1819.0,276.8,99.5,121.5,229.0,266.0,307.6,1879.5,9
Seasonal bread,292.0,66.9,32.4,7.1,41.6,62.9,86.2,172.9,0


In [4]:
# The three highest days per product group
top = (
    sales.sort_values("Umsatz", ascending=False)
    .groupby("Warengruppe")
    .head(3)
    .sort_values(["Warengruppe", "Umsatz"], ascending=[True, False])
)
top.assign(group=top["Warengruppe"].map(GROUPS), weekday=top["Datum"].dt.day_name())[["group", "Datum", "weekday", "Umsatz"]]

,group,Datum,weekday,Umsatz
1698,Bread,2018-03-31,Saturday,416.788764
1356,Bread,2017-04-15,Saturday,396.160770
1697,Bread,2018-03-29,Thursday,369.627890
2119,Rolls,2014-05-05,Monday,1203.431130
1852,Rolls,2013-08-03,Saturday,930.801703
2215,Rolls,2014-08-09,Saturday,874.906746
3938,Croissants,2014-05-05,Monday,565.941679
3671,Croissants,2013-08-03,Saturday,454.705781
4042,Croissants,2014-08-18,Monday,452.006266
5674,Confectionery,2014-02-23,Sunday,430.500089


## 3. Days without any sales (closed days)
We compare the full calendar with the dates that have at least one row.

In [5]:
calendar = pd.date_range(sales["Datum"].min(), sales["Datum"].max())
opening_days = pd.DatetimeIndex(sales["Datum"].unique())
closed = calendar.difference(opening_days)
print("Calendar days:", len(calendar), "| opening days:", len(opening_days), "| closed days:", len(closed))
pd.DataFrame({"Datum": closed.date, "weekday": closed.day_name()})

Calendar days: 1857 | opening days: 1819 | closed days: 38


,Datum,weekday
0,2013-12-25,Wednesday
1,2013-12-26,Thursday
2,2013-12-28,Saturday
3,2014-01-01,Wednesday
4,2014-04-18,Friday
5,2014-05-01,Thursday
6,2014-05-03,Saturday
7,2014-05-04,Sunday
8,2014-08-17,Sunday
9,2014-12-25,Thursday


26 of these 38 days are public holidays (1 Jan, Good Friday, 1 May, 25/26 Dec, Whit Monday 2018, Reformation Day 2017). The other 12 are treated as **unplanned closures** (DECISIONS.md). Closed days are not forecast.

## 4. Missing confectionery rows (product group 4)
Confectionery has fewer rows than the other main groups. We look at *when* the rows are missing and whether those days are otherwise normal. If missingness depends on something, it is not random and must not be filled naively.

In [6]:
conf_days = set(sales.loc[sales["Warengruppe"] == 4, "Datum"])
missing_conf = pd.DatetimeIndex(sorted(d for d in opening_days if d not in conf_days))
print("Opening days without confectionery:", len(missing_conf))
print("By year:", missing_conf.year.value_counts().sort_index().to_dict())
print("By month:", missing_conf.month.value_counts().sort_index().to_dict())

total_1_3 = sales[sales["Warengruppe"].isin([1, 2, 3])].groupby("Datum")["Umsatz"].sum()
print("Mean turnover of groups 1-3 on these days:", round(total_1_3.loc[missing_conf].mean(), 1))
print("Mean turnover of groups 1-3 on other days:", round(total_1_3.drop(missing_conf).mean(), 1))

Opening days without confectionery: 53
By year: {2013: 16, 2014: 23, 2015: 10, 2016: 4}
By month: {3: 1, 4: 4, 5: 4, 6: 5, 7: 11, 8: 4, 9: 3, 10: 1, 11: 4, 12: 16}
Mean turnover of groups 1-3 on these days: 860.7
Mean turnover of groups 1-3 on other days: 682.8


The gaps fall on busier-than-usual days, so they are not zero demand. Decision: these rows are not filled; reason unknown.

## 5. Seasonal bread (product group 6): season boundaries

In [7]:
s6 = sales.loc[sales["Warengruppe"] == 6, "Datum"]
season_start_year = s6.dt.year.where(s6.dt.month >= 10, s6.dt.year - 1).rename("season")
s6.groupby(season_start_year).agg(first_day="min", last_day="max", rows="count")

,first_day,last_day,rows
season,,,
2013,2013-10-24,2014-01-02,65
2014,2014-10-24,2015-01-06,66
2015,2015-11-02,2015-12-31,53
2016,2016-11-03,2016-12-29,52
2017,2017-10-30,2017-12-27,56


## 6. Weather
These are *observed* values, so they are not known 4–8 weeks ahead (canvas block 6). We profile them for the typical-weather feature and the upper-limit experiment. Wind speed is shown here but dropped from the project (doubtful unit: daily means up to 35 "m/s").

In [8]:
print("Shape:", weather.shape, "| duplicated dates:", weather["Datum"].duplicated().sum())
print("Period:", weather["Datum"].min().date(), "to", weather["Datum"].max().date())
print("Missing values per column:")
print(weather.isna().sum())
weather.describe().round(2)

Shape: (2601, 5) | duplicated dates: 0
Period: 2012-01-01 to 2019-08-01
Missing values per column:
Datum                    0
Bewoelkung              10
Temperatur               0
Windgeschwindigkeit      0
Wettercode             669
dtype: int64


,Datum,Bewoelkung,Temperatur,Windgeschwindigkeit,Wettercode
count,2601,2591.00,2601.00,2601.00,1932.00
mean,2015-12-07 00:08:18.269896,4.81,12.10,11.19,37.48
min,2012-01-01 00:00:00,0.00,-10.25,3.00,0.00
25%,2014-03-14 00:00:00,3.00,6.51,8.00,10.00
50%,2015-12-24 00:00:00,6.00,12.00,10.00,28.00
75%,2017-10-07 00:00:00,7.00,17.82,14.00,61.00
max,2019-08-01 00:00:00,8.00,32.67,35.00,95.00
std,NaN,2.58,7.17,4.12,27.57


In [9]:
weather_calendar = pd.date_range(weather["Datum"].min(), weather["Datum"].max())
missing_weather = weather_calendar.difference(weather["Datum"])
print("Dates without a weather row:", len(missing_weather))
in_sales_period = missing_weather[(missing_weather >= calendar[0]) & (missing_weather <= calendar[-1])]
print("... of which in the sales period:", [str(d.date()) for d in in_sales_period])
print("Missing cloud cover on:", weather.loc[weather["Bewoelkung"].isna(), "Datum"].dt.date.astype(str).tolist())

Dates without a weather row: 169
... of which in the sales period: ['2016-12-11', '2017-10-04', '2017-10-05']
Missing cloud cover on: ['2017-10-30', '2017-10-31', '2017-11-01', '2017-11-02', '2017-11-03', '2017-11-04', '2017-11-05', '2017-11-06', '2017-11-07', '2017-11-08']


Is a missing weather code random? We compare days with and without a code.

In [10]:
code_missing = weather["Wettercode"].isna().map({True: "code missing", False: "code present"})
weather.groupby(code_missing)[["Bewoelkung", "Temperatur"]].mean().round(2)

,Bewoelkung,Temperatur
Wettercode,,
code missing,3.29,15.44
code present,5.32,10.94


Days without a weather code are clearer and warmer: the code is most likely left out when nothing notable happened. Missing therefore carries information (decision: own category in the upper-limit experiment). Missing cloud cover and days without weather stay empty.

## 7. Kiel Week

In [11]:
print("Days per year:", kiel_week["Datum"].dt.year.value_counts().sort_index().to_dict())
kiel_week.groupby(kiel_week["Datum"].dt.year)["Datum"].agg(["min", "max"])

Days per year: {2012: 9, 2013: 9, 2014: 9, 2015: 9, 2016: 9, 2017: 9, 2018: 9, 2019: 9}


,min,max
Datum,,
2012,2012-06-16,2012-06-24
2013,2013-06-22,2013-06-30
2014,2014-06-21,2014-06-29
2015,2015-06-20,2015-06-28
2016,2016-06-18,2016-06-26
2017,2017-06-17,2017-06-25
2018,2018-06-16,2018-06-24
2019,2019-06-22,2019-06-30


## 8. Join
Decision (DECISIONS.md): **left join** of weather and Kiel Week onto sales on the date, so that every sales row stays. Days without a Kiel Week entry get 0. The row count must stay 9,334.

In [12]:
data = sales.merge(weather, on="Datum", how="left").merge(kiel_week, on="Datum", how="left")
data["KielerWoche"] = data["KielerWoche"].fillna(0).astype(int)
print("Rows after join:", len(data), "| rows in sales:", len(sales))
print("Missing values after join:")
print(data.isna().sum())
print("Kiel Week dates in the sales period:", data.loc[data["KielerWoche"] == 1, "Datum"].nunique())

Rows after join: 9334 | rows in sales: 9334
Missing values after join:
id                        0
Datum                     0
Warengruppe               0
Umsatz                    0
Bewoelkung               70
Temperatur               16
Windgeschwindigkeit      16
Wettercode             2325
KielerWoche               0
dtype: int64
Kiel Week dates in the sales period: 45


In [13]:
# Opening days per weekday, ordered Monday to Sunday
pd.Series(opening_days.day_name()).value_counts().reindex(WEEKDAYS)

Monday       258
Tuesday      262
Wednesday    263
Thursday     259
Friday       256
Saturday     261
Sunday       260
Name: count, dtype: int64

## Summary
- Sales: 9,334 rows, one per product group and opening day; key unique; no missing values; 1,819 opening days, 38 closed days.
- Confectionery has no row on 53 busier-than-usual days (not random): not filled.
- Seasonal-bread season start and end vary by year: season treated as known in advance.
- Weather: 3 sales dates without weather, 10 missing cloud-cover values (left empty), missing weather codes not random; wind speed dropped.
- Kiel Week: 5 Kiel Weeks (45 days) in the sales period.
- The left join keeps every sales row.

All decisions: `DECISIONS.md`. Checks that keep these facts true: `checks/`.